In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS cricketscoreprediction.bronze
COMMENT 'Bronze layer: Raw data'
""")
print("✓ Bronze schema created")


In [0]:
# Create the target table with SCD Type 2 columns if it doesn't exist
spark.sql("""
CREATE TABLE IF NOT EXISTS cricketscoreprediction.bronze.matches_info_structured (
  match_key STRING COMMENT 'Unique business key for the match',
  city STRING,
  dates ARRAY<STRING>,
  match_type STRING,
  teams ARRAY<STRING>,
  toss_decision STRING,
  toss_winner STRING,
  umpires ARRAY<STRING>,
  venue STRING,
  player_of_match ARRAY<STRING>,
  outcome_result STRING,
  outcome_winner STRING,
  outcome_by_runs BIGINT,
  outcome_by_wickets BIGINT,
  overs BIGINT,
  valid_from TIMESTAMP COMMENT 'SCD Type 2: Record valid from timestamp',
  valid_to TIMESTAMP COMMENT 'SCD Type 2: Record valid to timestamp',
  is_current BOOLEAN COMMENT 'SCD Type 2: Is this the current version',
  checksum STRING COMMENT 'Hash of all columns for change detection'
)
USING DELTA
COMMENT 'Bronze layer: Structured match information with SCD Type 2 history'
""")

print("✓ Target table created/verified: cricketscoreprediction.silver.matches_info_structured")

In [0]:
# Create a cleaned view for match info data
df_matches_structured = spark.sql("""
SELECT
  info AS info_struct
FROM cricketscoreprediction.bronze.matchesdata
WHERE info IS NOT NULL
""")

df_matches_structured.createOrReplaceTempView("matchesstructured")

print("✓ Temporary view 'matchesstructured' created")

In [0]:
# Prepare source data with match key and checksum
source_df = spark.sql("""
SELECT
  md5(concat_ws('|', 
    coalesce(info_struct.venue, ''),
    coalesce(array_join(info_struct.dates, ','), ''),
    coalesce(array_join(info_struct.teams, ','), '')
  )) AS match_key,
  info_struct.city,
  info_struct.dates,
  info_struct.match_type,
  info_struct.teams,
  info_struct.toss.decision AS toss_decision,
  info_struct.toss.winner AS toss_winner,
  info_struct.officials.umpires AS umpires,
  info_struct.venue,
  info_struct.player_of_match,
  info_struct.outcome['result'] AS outcome_result,
  info_struct.outcome.winner AS outcome_winner,
  info_struct.outcome.by.runs AS outcome_by_runs,
  info_struct.outcome.by.wickets AS outcome_by_wickets,
  info_struct.overs,
  md5(concat_ws('|',
    coalesce(info_struct.city, ''),
    coalesce(array_join(info_struct.dates, ','), ''),
    coalesce(info_struct.match_type, ''),
    coalesce(array_join(info_struct.teams, ','), ''),
    coalesce(info_struct.toss.decision, ''),
    coalesce(info_struct.toss.winner, ''),
    coalesce(array_join(info_struct.officials.umpires, ','), ''),
    coalesce(info_struct.venue, ''),
    coalesce(array_join(info_struct.player_of_match, ','), ''),
    coalesce(info_struct.outcome['result'], ''),
    coalesce(info_struct.outcome.winner, ''),
    coalesce(cast(info_struct.outcome.by.runs as string), ''),
    coalesce(cast(info_struct.outcome.by.wickets as string), ''),
    coalesce(cast(info_struct.overs as string), '')
  )) AS checksum
FROM matchesstructured
""")

source_df.createOrReplaceTempView("matches_source")

# MERGE with SCD Type 2 logic
spark.sql("""
MERGE INTO cricketscoreprediction.bronze.matches_info_structured AS target
USING matches_source AS source
ON target.match_key = source.match_key AND target.is_current = true

-- When match exists but data changed: expire old record and insert new
WHEN MATCHED AND target.checksum != source.checksum THEN
  UPDATE SET 
    target.valid_to = current_timestamp(),
    target.is_current = false

-- When match doesn't exist: insert new record
WHEN NOT MATCHED THEN
  INSERT (
    match_key, city, dates, match_type, teams, toss_decision, toss_winner,
    umpires, venue, player_of_match, outcome_result, outcome_winner,
    outcome_by_runs, outcome_by_wickets, overs,
    valid_from, valid_to, is_current, checksum
  )
  VALUES (
    source.match_key, source.city, source.dates, source.match_type, source.teams,
    source.toss_decision, source.toss_winner, source.umpires, source.venue,
    source.player_of_match, source.outcome_result, source.outcome_winner,
    source.outcome_by_runs, source.outcome_by_wickets, source.overs,
    current_timestamp(), null, true, source.checksum
  )
""")

# Insert new versions for changed records using a CTE to avoid correlated subquery
spark.sql("""
WITH latest_expired AS (
  SELECT 
    match_key,
    MAX(valid_to) as max_valid_to
  FROM cricketscoreprediction.bronze.matches_info_structured
  WHERE is_current = false AND valid_to IS NOT NULL
  GROUP BY match_key
),
changed_records AS (
  SELECT 
    source.match_key,
    source.city,
    source.dates,
    source.match_type,
    source.teams,
    source.toss_decision,
    source.toss_winner,
    source.umpires,
    source.venue,
    source.player_of_match,
    source.outcome_result,
    source.outcome_winner,
    source.outcome_by_runs,
    source.outcome_by_wickets,
    source.overs,
    source.checksum
  FROM matches_source AS source
  INNER JOIN cricketscoreprediction.bronze.matches_info_structured AS target
    ON source.match_key = target.match_key
    AND target.is_current = false
  INNER JOIN latest_expired
    ON target.match_key = latest_expired.match_key
    AND target.valid_to = latest_expired.max_valid_to
  WHERE source.checksum != target.checksum
)
INSERT INTO cricketscoreprediction.bronze.matches_info_structured
SELECT
  match_key,
  city,
  dates,
  match_type,
  teams,
  toss_decision,
  toss_winner,
  umpires,
  venue,
  player_of_match,
  outcome_result,
  outcome_winner,
  outcome_by_runs,
  outcome_by_wickets,
  overs,
  current_timestamp() AS valid_from,
  CAST(NULL AS TIMESTAMP) AS valid_to,
  true AS is_current,
  checksum
FROM changed_records
""")

print("✓ Matches info data merged with SCD Type 2")

In [0]:
# Create the target table for innings with SCD Type 2 columns
spark.sql("""
CREATE TABLE IF NOT EXISTS cricketscoreprediction.bronze.matches_innings_structured (
  innings_key STRING COMMENT 'Unique business key for the innings',
  team STRING,
  super_over BOOLEAN,
  target_overs DOUBLE,
  target_runs BIGINT,
  powerplay_from DOUBLE,
  powerplay_to DOUBLE,
  powerplay_type STRING,
  over_number BIGINT,
  delivery STRUCT<
    actual_delivery: STRING,
    batter: STRING,
    bowler: STRING,
    extras: STRUCT<byes: BIGINT, legbyes: BIGINT, noballs: BIGINT, penalty: BIGINT, wides: BIGINT>,
    non_striker: STRING,
    replacements: STRUCT<match: ARRAY<STRUCT<in: STRING, out: STRING, reason: STRING, team: STRING>>, role: ARRAY<STRUCT<in: STRING, out: STRING, reason: STRING, role: STRING>>>,
    review: STRUCT<batter: STRING, by: STRING, decision: STRING, type: STRING, umpire: STRING, umpires_call: BOOLEAN>,
    runs: STRUCT<batter: BIGINT, extras: BIGINT, non_boundary: BOOLEAN, total: BIGINT>,
    wickets: ARRAY<STRUCT<fielders: ARRAY<STRUCT<name: STRING, substitute: BOOLEAN>>, kind: STRING, player_out: STRING>>
  >,
  valid_from TIMESTAMP COMMENT 'SCD Type 2: Record valid from timestamp',
  valid_to TIMESTAMP COMMENT 'SCD Type 2: Record valid to timestamp',
  is_current BOOLEAN COMMENT 'SCD Type 2: Is this the current version',
  checksum STRING COMMENT 'Hash of all columns for change detection'
)
USING DELTA
COMMENT 'Bronze layer: Structured innings and delivery data with SCD Type 2 history'
""")

print("✓ Target table created/verified: cricketscoreprediction.bronze.matches_innings_structured")

In [0]:
# Create a cleaned view for innings data
df_innings_structured = spark.sql("""
SELECT
  EXPLODE(innings) AS inning
FROM cricketscoreprediction.bronze.matchesdata
WHERE innings IS NOT NULL
""")

df_innings_structured.createOrReplaceTempView("inningsstructured")

print("✓ Temporary view 'inningsstructured' created")

In [0]:
# Prepare source data with innings key and checksum
# Use ROW_NUMBER to create unique key for each delivery
source_innings_df = spark.sql("""
WITH numbered_deliveries AS (
  SELECT
    inning,
    powerplay,
    over_data,
    delivery,
    ROW_NUMBER() OVER (
      PARTITION BY inning.team, over_data.over, delivery.batter, delivery.bowler, delivery.actual_delivery
      ORDER BY powerplay.from, powerplay.to
    ) AS delivery_seq
  FROM inningsstructured
  LATERAL VIEW OUTER explode(inning.powerplays) AS powerplay
  LATERAL VIEW OUTER explode(inning.overs) AS over_data
  LATERAL VIEW OUTER explode(over_data.deliveries) AS delivery
)
SELECT
  md5(concat_ws('|',
    coalesce(inning.team, ''),
    coalesce(cast(over_data.over as string), ''),
    coalesce(delivery.batter, ''),
    coalesce(delivery.bowler, ''),
    coalesce(delivery.actual_delivery, ''),
    coalesce(cast(delivery_seq as string), '')
  )) AS innings_key,
  inning.team,
  inning.super_over,
  inning.target.overs AS target_overs,
  inning.target.runs AS target_runs,
  powerplay.from AS powerplay_from,
  powerplay.to AS powerplay_to,
  powerplay.type AS powerplay_type,
  over_data.over AS over_number,
  delivery,
  md5(to_json(struct(
    inning.team,
    inning.super_over,
    inning.target.overs,
    inning.target.runs,
    powerplay.from,
    powerplay.to,
    powerplay.type,
    over_data.over,
    delivery
  ))) AS checksum
FROM numbered_deliveries
""")

source_innings_df.createOrReplaceTempView("innings_source")

# MERGE with SCD Type 2 logic
spark.sql("""
MERGE INTO cricketscoreprediction.bronze.matches_innings_structured AS target
USING innings_source AS source
ON target.innings_key = source.innings_key AND target.is_current = true

-- When innings exists but data changed: expire old record
WHEN MATCHED AND target.checksum != source.checksum THEN
  UPDATE SET 
    target.valid_to = current_timestamp(),
    target.is_current = false

-- When innings doesn't exist: insert new record
WHEN NOT MATCHED THEN
  INSERT (
    innings_key, team, super_over, target_overs, target_runs,
    powerplay_from, powerplay_to, powerplay_type, over_number, delivery,
    valid_from, valid_to, is_current, checksum
  )
  VALUES (
    source.innings_key, source.team, source.super_over, source.target_overs,
    source.target_runs, source.powerplay_from, source.powerplay_to,
    source.powerplay_type, source.over_number, source.delivery,
    current_timestamp(), null, true, source.checksum
  )
""")

# Insert new versions for changed records using a CTE to avoid correlated subquery
spark.sql("""
WITH latest_expired AS (
  SELECT 
    innings_key,
    MAX(valid_to) as max_valid_to
  FROM cricketscoreprediction.bronze.matches_innings_structured
  WHERE is_current = false AND valid_to IS NOT NULL
  GROUP BY innings_key
),
changed_records AS (
  SELECT 
    source.innings_key,
    source.team,
    source.super_over,
    source.target_overs,
    source.target_runs,
    source.powerplay_from,
    source.powerplay_to,
    source.powerplay_type,
    source.over_number,
    source.delivery,
    source.checksum
  FROM innings_source AS source
  INNER JOIN cricketscoreprediction.bronze.matches_innings_structured AS target
    ON source.innings_key = target.innings_key
    AND target.is_current = false
  INNER JOIN latest_expired
    ON target.innings_key = latest_expired.innings_key
    AND target.valid_to = latest_expired.max_valid_to
  WHERE source.checksum != target.checksum
)
INSERT INTO cricketscoreprediction.bronze.matches_innings_structured
SELECT
  innings_key,
  team,
  super_over,
  target_overs,
  target_runs,
  powerplay_from,
  powerplay_to,
  powerplay_type,
  over_number,
  delivery,
  current_timestamp() AS valid_from,
  CAST(NULL AS TIMESTAMP) AS valid_to,
  true AS is_current,
  checksum
FROM changed_records
""")

print("✓ Innings data merged with SCD Type 2")